In [ ]:
%load_ext autoreload
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
import BaselineModel


In [ ]:
load_dotenv()
DEFAULT_DETECTION_CLASS = 'no'

In [ ]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)


In [ ]:
import shutil
BASE_MAT_DIRECTORY = os.getenv('BASE_MAT_DIRECTORY')
MAT_NEW_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/new/input'
MAT_NEW_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/new/output'
shutil.copytree('../config/baseline/dic', MAT_NEW_INPUT_DIRECTORY + '/dic', dirs_exist_ok=True)
os.makedirs(os.path.join(MAT_NEW_INPUT_DIRECTORY, 'origin'), exist_ok=True)
os.makedirs(MAT_NEW_OUTPUT_DIRECTORY, exist_ok=True)
for kv in [{'train': detect_train_df}, {'test': detect_test_df}, {'merged': pd.concat([detect_train_df.assign(project='train'), detect_test_df.assign(project='test')])}]:
    for df_name, df in kv.items():
        if df_name == 'merged':
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/comments', index=False, header=False)
            df['label'].str.lower().map({'yes': 'SATD', 'no': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/labels', index=False, header=False)
            df['project'].to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/projects', index=False, header=False)
        else:
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/data--{df_name}.txt', index=False, header=False)
            df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/label--{df_name}.txt', index=False, header=False)




MAT_PRETRAINED_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/input'
MAT_PRETRAINED_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/output'
os.makedirs(MAT_PRETRAINED_OUTPUT_DIRECTORY, exist_ok=True)
shutil.copytree('../config/baseline', MAT_PRETRAINED_INPUT_DIRECTORY, dirs_exist_ok=True)

detect_test_df['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/data--test.txt', index=False, header=False)
detect_test_df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', index=False, header=False)
df1 = pd.DataFrame(open('../config/baseline/origin/data--train.txt').read().splitlines(), columns=["text"])
df2 = detect_test_df[['text']]
pd.concat([df1, df2])['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/comments', index=False, header=False)

df1 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--train.txt', header=None, names=['label']).assign(project='train')
df2 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', header=None, names=['label']).assign(project='test')
df = pd.concat([df1, df2])
df['label'].map({'positive': 'SATD', 'negative': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/labels', columns=['label'], index=False, header=False)

df.to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/projects', columns=['project'], index=False, header=False)



In [ ]:
for mode in ['pretrained', 'new']:
    pattern_model = BaselineModel('detect', f'{mode}-Pattern', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    pattern_model.fit(detect_train_dataset)
    pattern_model.predict(detect_test_dataset)

In [ ]:
for mode in ['pretrained', 'new']:
    tm_model = BaselineModel('detect', f'{mode}-TM', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    tm_model.fit(detect_train_dataset)
    tm_model.predict(detect_test_dataset)

In [ ]:
for mode in ['pretrained', 'new']:
    nlp_model = BaselineModel('detect', f'{mode}-NLP', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    nlp_model.fit(detect_train_dataset)
    nlp_model.predict(detect_test_dataset)

In [ ]:
for mode in ['pretrained', 'new']:
    mat_model = BaselineModel('detect', f'{mode}-MAT', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    mat_model.fit(detect_train_dataset)
    mat_model.predict(detect_test_dataset)